In [ ]:
import cuml
import cupy
import cudf
import cugraph
import cuspatial
import cupyx
import cupyx.scipy.sparse.linalg

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib
from tqdm import tqdm
import itertools
import os
import numpy as np
import timeit
from scipy.cluster import hierarchy
import scipy.sparse
import scanpy as sc
import pandas as pd
import re
import random
sc.set_figure_params(dpi=80)

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import logging
logging.getLogger('matplotlib.font_manager').setLevel(level=logging.CRITICAL)

In [ ]:
import importlib.util
import sys

def lazy_import(module_name, path_to_file):
    spec = importlib.util.spec_from_file_location(module_name,path_to_file)
    foo = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = foo
    spec.loader.exec_module(foo)
    return foo

utils_dir = "../../utils"
general_utils =  lazy_import("general_utils",os.path.join(utils_dir, "general_utils.py"))
preprocessing_utils = lazy_import("preprocessing_utils",os.path.join(utils_dir, "preprocessing_utils.py"))

from general_utils import ismember, grep, grep_exclude
from preprocessing_utils import kneepoint

Diffusion component functions from Setty et al 2018

In [ ]:
from scipy.sparse import csr_matrix, find, issparse
from scipy.sparse.linalg import eigs
import matplotlib

def get_diffusion_operator_cpu(adata):
    print("Computing diffusion kernel using CPU")
    N = adata.n_obs
    k = adata.obsm['indices'].shape[1]
    adaptive_k = int(np.floor(k / 3))
    adaptive_std = np.zeros(N)
    
    for i in np.arange(len(adaptive_std)):
        adaptive_std[i] = np.sort(adata.obsm["distances"][i,:])[adaptive_k - 1]

    # Kernel
    targets = np.ravel(adata.obsm['indices'])
    sources = np.ravel([[i]*k for i in range(adata.n_obs)])
    dists = np.ravel(adata.obsm['distances'])

    # X, y specific stds
    dists = dists / adaptive_std[sources]
    W = scipy.sparse.csr_matrix((np.exp(-dists), (sources, targets)), shape=[N, N])
    
    # Diffusion components
    kernel = W + W.T
    
    print("Computing diffusion operator using CPU")
    # Markov
    D = np.ravel(kernel.sum(axis=1))
    D[D != 0] = 1 / D[D != 0]
    D = scipy.sparse.csr_matrix((D, (range(N), range(N))), shape=[N, N])
    T = D.dot(kernel)
    
    return(T)

def getDiffusionComponents_cpu(adata, n_components = 10):
    N = adata.n_obs
    T = get_diffusion_operator_cpu(adata)

    seed = 0
    np.random.seed(seed)
    v0 = np.random.rand(min(T.shape))
    # Eigen value dcomposition
    print("Getting diffusion components using CPU")
    D, V = scipy.sparse.linalg.eigs(T, n_components, tol=1e-4, maxiter=1000, v0 = v0)
    D = np.real(D)
    V = np.real(V)
    inds = np.argsort(D)[::-1]
    D = D[inds]
    V = V[:,inds]

    # Normalize
    for i in range(V.shape[1]):
        V[:, i] = V[:, i] / np.linalg.norm(V[:, i])

    # Create are results dictionary
    adata.obsp['diff_operator'] = T.copy()
    #adata.obsm['X_diff_comp'] = V[:,1:].copy()
    #adata.uns['diff_comp_eigenvalues'] = D[1:].copy()
    adata.obsm['X_diff_comp'] = V.copy()
    adata.uns['diff_comp_eigenvalues'] = D.copy()
    return(adata)

def plot_diffusion_components(tsne, DCs, vmax=None):
    """ Plots the diffusion components on tSNE maps
    :return: fig, ax
    """

    # Please run tSNE before plotting diffusion components. #
    # Please run diffusion maps using run_diffusion_map before plotting #

    # Plot
    fig = FigureGrid(DCs.shape[1], 5)

    for i, ax in enumerate(fig):
        ax.scatter(
            tsne[:,0],
            tsne[:,1],
            c=DCs[:, i],
            cmap=matplotlib.cm.Spectral_r,
            edgecolors="none",
            s=3,
        )
        ax.xaxis.set_major_locator(plt.NullLocator())
        ax.yaxis.set_major_locator(plt.NullLocator())
        ax.set_aspect("equal")
        ax.set_title("Component %d" % i, fontsize=10)
        ax.set_axis_off()

class FigureGrid:
    """
    Generates a grid of axes for plotting
    axes can be iterated over or selected by number. e.g.:
    >>> # iterate over axes and plot some nonsense
    >>> fig = FigureGrid(4, max_cols=2)
    >>> for i, ax in enumerate(fig):
    >>>     plt.plot(np.arange(10) * i)
    >>> # select axis using indexing
    >>> ax3 = fig[3]
    >>> ax3.set_title("I'm axis 3")
    """

    # Figure Grid is favorable for displaying multiple graphs side by side.

    def __init__(self, n: int, max_cols=3, scale=3):
        """
        :param n: number of axes to generate
        :param max_cols: maximum number of axes in a given row
        """

        self.n = n
        self.nrows = int(np.ceil(n / max_cols))
        self.ncols = int(min((max_cols, n)))
        figsize = self.ncols * scale, self.nrows * scale

        # create figure
        self.gs = plt.GridSpec(nrows=self.nrows, ncols=self.ncols)
        self.figure = plt.figure(figsize=figsize)

        # create axes
        self.axes = {}
        for i in range(n):
            row = int(i // self.ncols)
            col = int(i % self.ncols)
            self.axes[i] = plt.subplot(self.gs[row, col])

    def __getitem__(self, item):
        return self.axes[item]

    def __iter__(self):
        for i in range(self.n):
            yield self[i]

### Import dataset

In [ ]:
output_path = '../xenium_preprocessing_outputs'

knn_output_path = os.path.join(output_path, 'knn')
pca_output_path = os.path.join(output_path, 'pca')
umap_output_path = os.path.join(output_path, 'umap')
adata_base_output_path = os.path.join(output_path, 'adata_base')
obs_output_path = os.path.join(output_path, 'obs')
updated_obs_output_path = os.path.join(output_path, 'obs_updated')
compiled_output_path = os.path.join(output_path, 'compiled')
seacells_output_path = os.path.join(output_path, 'seacells')
diffusion_output_path = os.path.join(output_path, 'diffusion')

os.makedirs(diffusion_output_path, exist_ok=True)


In [ ]:
def get_compiled_adata_subset(selected_tier, selected_subset, use_log, explained_var, umap_n_neighbors, cluster_n_neighbors, selected_layer = 'raw_in_nucleus', umap_min_dist = 0.05):
    dataset_prefix = "adata_" + selected_tier + "_SUBSET_%s_LAYER_%s_LOG_%s" % (selected_subset, selected_layer, use_log)
    file_prefix = "%s_EXPLAINEDVAR_%d_KNN_%s" % (dataset_prefix, int(explained_var * 100), "%d")
    
    adata_base_file = os.path.join(adata_base_output_path, dataset_prefix + ".h5ad")
    obs_file = os.path.join(obs_output_path,file_prefix % (cluster_n_neighbors) + "__obs.h5ad" )
    umap_file = os.path.join(umap_output_path,file_prefix % (umap_n_neighbors) + "_MINDIST_%d__umap.csv.gz" % int(umap_min_dist * 100))
    knn_file = os.path.join(knn_output_path,file_prefix % (cluster_n_neighbors) + "__knn.h5ad" )
    pca_file = os.path.join(pca_output_path, "%s_EXPLAINEDVAR_%d__pca.h5ad" % (dataset_prefix, int(explained_var * 100)))
    
    output_template = dataset_prefix + "_EXPLAINEDVAR_%d_CLUSTERKNN_%d__diffusion.h5ad"
    output_file = os.path.join(diffusion_output_path, output_template % (int(explained_var * 100), cluster_n_neighbors))   
    
    print("Reading counts file")
    adata = sc.read_h5ad(adata_base_file)
    print("Reading obs file")
    obs = sc.read_h5ad(obs_file)
    print("Reading umap file")
    umap = pd.read_csv(umap_file, index_col = 0)
    print("Reading knn file")
    knn = sc.read_h5ad(knn_file)    
    print("Reading pca file")
    pca = sc.read_h5ad(pca_file)
    
    adata.obsm['indices'] = knn.obsm['indices'].copy()
    adata.obsm['distances'] = knn.obsm['distances'].copy()
    adata.obsm['X_pca'] = pca.X.copy()
    
    adata.obsm['X_umap'] = umap.to_numpy()
    for colname in obs.obs.columns:
        adata.obs[colname] = pd.Categorical(obs.obs[colname])

    master_obs_file = "adata_tier0_SUBSET_all_LAYER_%s_LOG_%s_EXPLAINEDVAR_%d_KNN_%s__obs.h5ad" % (selected_layer, use_log, int(explained_var * 100), cluster_n_neighbors)
    master_obs_file = os.path.join(updated_obs_output_path, master_obs_file)
    print("Inheriting master annotations")
    master_obs = sc.read_h5ad(master_obs_file)
    if np.sum(np.logical_not(ismember(adata.obs_names, master_obs.obs_names)[1]) == 0):
        master_obs = master_obs[adata.obs_names,:].copy()
        adata.obs['cell_type_1'] = master_obs.obs['cell_type_1'].copy()
        adata.obs['cell_type_0'] = master_obs.obs['cell_type_0'].copy()
    
    return(adata, output_file)

In [ ]:
subset_dictionary = {
    'Premalignant0': ['progenitor1', 'progenitor2', 'gastricprogenitor', 'gastric', 'gastric_pit', 'gastric_chief', 'gastric_Ccn2'],
    'Premalignant1': ['progenitor1', 'progenitor2', 'gastricprogenitor', 'gastric', 'gastric_pit', 'gastric_chief'],
    'Premalignant2': ['progenitor1', 'progenitor2', 'gastricprogenitor', 'gastric_pit', 'gastric_chief', 'adm', 'gastric'],
    'Premalignant3': ['progenitor1', 'progenitor2', 'gastricprogenitor'],
    'Premalignant4': ['progenitor1', 'progenitor2', 'gastricprogenitor', 'adm'],
    'Premalignant5': ['progenitor1', 'progenitor2'],
    'Premalignant6': ['progenitor1', 'progenitor2', 'adm'],
    'Premalignant7': ['adm', 'gastric', 'gastric_pit', 'gastric_chief'],
    'Premalignant8': ['adm', 'gastric', 'gastric_pit', 'gastric_chief', 'gastric_Ccn2']
}

In [ ]:
selected_tier = 'tier3'
selected_subset = 'Premalignant0'
use_log = False
explained_var = 0.75
umap_n_neighbors = 10
cluster_n_neighbors = 30
selected_layer = 'raw_in_nucleus'
umap_min_dist = 0.1
adata, output_file = get_compiled_adata_subset(selected_tier = selected_tier, selected_subset = selected_subset, use_log = use_log, explained_var = explained_var, umap_n_neighbors = umap_n_neighbors, cluster_n_neighbors = cluster_n_neighbors, selected_layer = selected_layer, umap_min_dist = umap_min_dist)


Reading counts file
Reading obs file
Reading umap file
Reading knn file
Reading pca file
Inheriting master annotations


### Diffusion components

In [ ]:
adata = getDiffusionComponents_cpu(adata, n_components=20)

Computing diffusion kernel using CPU
Computing diffusion operator using CPU
Getting diffusion components using CPU


In [ ]:
plt.plot(adata.uns['diff_comp_eigenvalues'], 'o-')
plt.xlabel("DC")
plt.ylabel("EigenValue")

n_DCs = 7
plt.plot([n_DCs-0.5] * 2, plt.gca().get_ylim(), 'k--')

In [ ]:
import random
random_cells = random.sample(list(range(adata.n_obs)), int(adata.n_obs / 10))
adata_sub = adata[random_cells,:].copy()
plot_diffusion_components(adata_sub.obsm["X_umap"], adata_sub.obsm["X_diff_comp"][:,:n_DCs])

In [ ]:
adata_output = sc.AnnData(scipy.sparse.csr_matrix([[0] * n_DCs] * adata.n_obs))
adata_output.obsm['X_diff_comp'] = adata.obsm['X_diff_comp'][:,1:(n_DCs)]
adata_output.uns['diff_comp_eigenvalues'] = adata.uns['diff_comp_eigenvalues'][1:(n_DCs)]
adata_output.obsp['diff_operator'] = adata.obsp['diff_operator']
adata_output.obs_names = adata.obs_names
adata_output.obs['progenitor_DC'] = -adata_output.obsm['X_diff_comp'][:,0].copy()
sc.write(output_file, adata = adata_output)